# 1. Environment Setup & Library Imports
- 분석에 필요한 표준 라이브러리, 지리공간(GeoPandas), 수치 계산(NumPy/Pandas) 모듈 로드

In [18]:
import os
import re
import json
import math
import unicodedata
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 2. Configuration & Unified Scenario Definitions
- 경로 및 분석 파라미터(Cutoff 15분, 가우시안 감쇄) 설정
- 주간(오전/낮) 및 심야(00~06시) 통합 총 24개 시나리오(4개년 × 6개 조건) 정의

In [19]:
# ----------------------------------------------------
# [Cell 2] 표준 시나리오 정의 (기준표 5개 시나리오 반영)
# ----------------------------------------------------
# 시나리오 구조: (daytype, period, traffic_condition, (window_start, window_end), demand_col)

SCENARIOS = [
    # 1. 평일 오전 (7~9시) - congested
    ("week", "오전", "congested", (7, 9), "오전_avg"),
    
    # 2. 평일 낮 (11~13시) - normal
    ("week", "낮", "normal", (11, 13), "낮_avg"),
    
    # 3. 주말 오전 (7~9시) - freeflow
    ("weekend", "오전", "freeflow", (7, 9), "오전_avg"),
    
    # 4. 주말 낮 (11~13시) - normal
    ("weekend", "낮", "normal", (11, 13), "낮_avg"),
    
    # 5. 심야 (00~05시) - freeflow (평일/주말 각각 산출 후 비교 또는 단일 사용)
    ("week", "심야", "freeflow", (0, 5), "심야_avg"),
    ("weekend", "심야", "freeflow", (0, 5), "심야_avg")
]

# 3. Define Gaussian Distance Decay & Helper Functions
$$f_{\text{Gaussian}}(t) = \begin{cases} \dfrac{e^{-\frac{1}{2}(t/d_0)^2} - e^{-\frac{1}{2}}}{1 - e^{-\frac{1}{2}}}, & \text{if } t \le d_0 \\ 0, & \text{if } t > d_0 \end{cases}$$

In [20]:
TIME_RANGE_RE = re.compile(r"(\d{1,2})[:시](\d{2})?\s*[~-]\s*(\d{1,2})[:시](\d{2})?")

def parse_open_window(text):
    if not text or not str(text).strip() or "24시간" in str(text) or "24시" in str(text):
        return None
    t = str(text).strip()
    m = TIME_RANGE_RE.search(t)
    if not m:
        return None
    h1, _, h2, _ = m.groups()
    start, end = int(h1), int(h2)
    weekday_only = ("평일" in t) or ("주중" in t)
    return (start, end, weekday_only)

def is_open(parsed, window_start, window_end, daytype):
    if parsed is None:
        return True
    start, end, weekday_only = parsed
    if weekday_only and daytype == "weekend":
        return False
    if end <= start:  # 24시간 또는 익일 새벽까지 운영 (예: 20시 ~ 06시)
        return True
    return not (end <= window_start or start >= window_end)

def decay_gaussian(tt, d0=CUTOFF_SEC):
    tt = np.asarray(tt, dtype=np.float64)
    w = (np.exp(-0.5 * (tt / d0)**2) - math.exp(-0.5)) / (1.0 - math.exp(-0.5))
    return np.where(tt <= d0, w, 0.0)

# 4. G2SFCA Core Algorithm Implementation
- **Step 1:** 공급처(충전소 $j$) 중심의 가중 수급비 산출
  $$R_j = \frac{S_j}{\sum_{k \in \{t_{kj} \le d_0\}} D_k \cdot f_{\text{Gaussian}}(t_{kj})}$$
- **Step 2:** 수요처(집계구 $i$) 중심의 최종 접근성 산출
  $$A_i = \sum_{j \in \{t_{ij} \le d_0\}} R_j \cdot f_{\text{Gaussian}}(t_{ij})$$

In [21]:
def compute_g2sfca(df_od, dict_supply, s_demand, d0=CUTOFF_SEC):
    # 1. 거리 감쇄 가중치 계산 (NumPy 벡터화)
    w = decay_gaussian(df_od["travel_time_sec"].values, d0=d0)
    
    # 2. OD별 수요 매핑 및 Step 1 가중수요 산출
    demand_vals = df_od["oa_code"].map(s_demand).fillna(0.0).values
    w_demand = demand_vals * w
    
    df_temp_d = pd.DataFrame({
        "station_id": df_od["station_id"].values,
        "w_demand": w_demand
    })
    d_sum_series = df_temp_d.groupby("station_id")["w_demand"].sum()
    
    # 3. 공급 대 수요 비율 R_j 산출
    supply_series = pd.Series(dict_supply)
    r_j = (supply_series / d_sum_series).replace([np.inf, -np.inf], 0.0).fillna(0.0)
    
    # 4. Step 2 집계구별 최종 접근성 점수 A_i 산출
    r_vals = df_od["station_id"].map(r_j).fillna(0.0).values
    w_supply = r_vals * w
    
    df_temp_a = pd.DataFrame({
        "oa_code": df_od["oa_code"].values,
        "w_supply": w_supply
    })
    a_i = df_temp_a.groupby("oa_code")["w_supply"].sum()
    
    # 전체 집계구 인덱스 기준 정렬 및 결측치 0.0 보정
    return a_i.reindex(s_demand.index, fill_value=0.0)

# 5. Load Demand Data & Private Infrastructure Flags
- 아파트 단지 내 사유 충전기 목록 추출 (`is_apt_v3`)
- 연도별/시간대별 서울시 집계구 생활인구 메모리 사전 캐싱 (I/O 병목 해소)

In [22]:
# 1. 아파트 충전소 제외 ID Set 로드 (v3)
df_apt = pd.read_csv(APT_FP, dtype={"station_id": str})
apt_set = set(df_apt[df_apt["is_apt_v3"]]["station_id"])
print(f">> 아파트 제외 대상 충전소: {len(apt_set):,}개")

# 2. 생활인구 원본 데이터 로드
df_pop_all = pd.read_csv(D1_FP, dtype={"집계구코드": str})

def load_demand_series(year, col_name):
    df_y = df_pop_all[df_pop_all["year"] == year]
    return pd.Series(df_y[col_name].astype(float).values, index=df_y["집계구코드"].values)

# 3. 연도별 충전기 공급 데이터 로드
def load_supply_and_hours(year):
    fname = f"metro7_ev_chargers_{year}_fastonly.geojson"
    fp = unicodedata.normalize("NFD", str(CHARGER_DIR_FASTONLY / fname))
    with open(fp, encoding="utf-8") as f:
        data = json.load(f)
        
    supply, hours = {}, {}
    for feat in data["features"]:
        p = feat["properties"]
        if p.get("city") == "서울특별시":
            sid = str(p["station_id"])
            supply[sid] = float(p.get("fast_count", 0) or 0)
            hours[sid] = parse_open_window(p.get("openinghour", ""))
    return supply, hours

>> 아파트 제외 대상 충전소: 5,634개


# 6. Batch Execution Pipeline (Daytime + Simya)
- 연도별 공급 데이터, 아파트 제외(v3), 운영시간 필터링 적용 후 G2SFCA 일괄 산출

In [23]:
print("=" * 70)
print("RUNNING: UNIFIED GAUSSIAN G2SFCA PIPELINE (DAYTIME + SIMYA)")
print("=" * 70)

results_summary = []

for year in YEARS:
    raw_supply, hours_dict = load_supply_and_hours(year)
    print(f"\n[YEAR {year}] 서울시 급속충전소 {len(raw_supply)}개 로드 완료")
    
    for daytype, period, traffic, (w_start, w_end), pop_col in SCENARIOS:
        tag = f"{year}_{daytype}_{period}_{traffic}"
        
        # 1. 생활인구 수요 로드
        s_demand = load_demand_series(year, pop_col)
        
        # 2. 유효 공급량 산출 (아파트 제외 + 운영시간 필터)
        effective_supply = {}
        cnt_closed = 0
        for sid, count in raw_supply.items():
            if sid in apt_set:
                effective_supply[sid] = 0.0
                continue
            if not is_open(hours_dict.get(sid), w_start, w_end, daytype):
                effective_supply[sid] = 0.0
                cnt_closed += 1
            else:
                effective_supply[sid] = count
                
        # 3. OD Matrix 로드
        fp_od = OD_DIR / f"od_{tag}.csv"
        if not fp_od.exists():
            print(f"  [!] OD 파일 누락: {fp_od.name}")
            continue
            
        df_od = pd.read_csv(fp_od, dtype={"station_id": str, "oa_code": str})
        
        # 4. G2SFCA 계산
        s_score = compute_g2sfca(df_od, effective_supply, s_demand, d0=CUTOFF_SEC)
        
        # 5. 결과 CSV 저장 (_mw 적용)
        fp_out = DIR_OUTPUT / f"g2sfca_score_{tag}_mw.csv"
        df_result = pd.DataFrame({
            "oa_code": s_score.index,
            "accessibility_score": s_score.values
        })
        df_result.to_csv(fp_out, index=False, encoding="utf-8-sig")
        
        mean_permil = s_score.mean() * 1000
        print(f"  [>] {tag:<35} | Mean: {mean_permil:7.4f} ‰ | Closed: {cnt_closed:3d} EA")
        results_summary.append({
            "year": year,
            "daytype": daytype,
            "period": period,
            "traffic": traffic,
            "mean_permille": mean_permil
        })

RUNNING: UNIFIED GAUSSIAN G2SFCA PIPELINE (DAYTIME + SIMYA)

[YEAR 2021] 서울시 급속충전소 687개 로드 완료
  [>] 2021_week_오전_congested              | Mean:  0.0639 ‰ | Closed:  97 EA
  [>] 2021_week_낮_normal                  | Mean:  0.0795 ‰ | Closed:   2 EA
  [>] 2021_weekend_오전_freeflow            | Mean:  0.0655 ‰ | Closed: 113 EA
  [>] 2021_weekend_낮_normal               | Mean:  0.0755 ‰ | Closed:  38 EA
  [>] 2021_week_심야_freeflow               | Mean:  0.0648 ‰ | Closed: 123 EA
  [>] 2021_weekend_심야_freeflow            | Mean:  0.0648 ‰ | Closed: 123 EA

[YEAR 2022] 서울시 급속충전소 959개 로드 완료
  [>] 2022_week_오전_congested              | Mean:  0.1159 ‰ | Closed: 139 EA
  [>] 2022_week_낮_normal                  | Mean:  0.1459 ‰ | Closed:   4 EA
  [>] 2022_weekend_오전_freeflow            | Mean:  0.1191 ‰ | Closed: 158 EA
  [>] 2022_weekend_낮_normal               | Mean:  0.1396 ‰ | Closed:  51 EA
  [>] 2022_week_심야_freeflow               | Mean:  0.1169 ‰ | Closed: 176 EA
  [>] 2022_weekend_심야_fre

# 7. Summary Statistics & Result Inspection
- 최종 산출된 2024년 최신 시나리오의 기초 통계량 확인

In [24]:
df_summary = pd.DataFrame(results_summary)
piv_table = df_summary.pivot_table(
    index=["year", "period"], 
    columns="daytype", 
    values="mean_permille"
)
piv_table["차이(%)"] = (piv_table["weekend"] / piv_table["week"] - 1) * 100

print("\n=== [시간대별 평일 vs 주말 접근성 평균 (단위: ‰)] ===")
display(piv_table)


=== [시간대별 평일 vs 주말 접근성 평균 (단위: ‰)] ===


daytype       week  weekend   차이(%)
year period                        
2021 낮      0.0795   0.0755 -5.0071
     심야     0.0648   0.0648  0.0877
     오전     0.0639   0.0655  2.4668
2022 낮      0.1459   0.1396 -4.3007
     심야     0.1169   0.1170  0.1055
     오전     0.1159   0.1191  2.7089
2023 낮      0.2777   0.2700 -2.7788
     심야     0.2319   0.2325  0.2706
     오전     0.2297   0.2392  4.1546
2024 낮      0.3673   0.3589 -2.2931
     심야     0.3096   0.3107  0.3627
     오전     0.3086   0.3195  3.5332

# 8. Result Validation (Comparison with Original Outputs)
- 원본 산출물과 신규 산출물(_mw) 간 부동소수점 오차 검증

In [25]:
# [Cell 8] 원본 산출물 vs 신규 산출물(_mw) 전수 오차 정밀 검증
validation_records = []
DIR_SIMYA_ORIG = BASE_DIR / "output/g2sfca_sfast_simya_gaussian"

for year in YEARS:
    for daytype, period, traffic, _, _ in SCENARIOS:
        tag = f"{year}_{daytype}_{period}_{traffic}"
        fp_new = DIR_OUTPUT / f"g2sfca_score_{tag}_mw.csv"
        
        # 1. 원본 파일 경로 매핑 (심야 / 주간 정규 / 태그 없는 구버전 순차 탐색)
        if period == "심야":
            fp_orig = DIR_SIMYA_ORIG / f"g2sfca_score_{year}_{daytype}_심야.csv"
        else:
            fp_orig = DIR_OUTPUT / f"g2sfca_score_{tag}.csv"
            # _normal 또는 _traffic 태그가 없는 구버전 파일명 대비
            if not fp_orig.exists():
                fp_orig_alt = DIR_OUTPUT / f"g2sfca_score_{year}_{daytype}_{period}.csv"
                if fp_orig_alt.exists():
                    fp_orig = fp_orig_alt

        if not fp_new.exists():
            continue
            
        if not fp_orig.exists():
            validation_records.append({
                "Scenario": tag,
                "Status": "원본 파일 없음 (비교 스킵)",
                "Max Abs Diff": np.nan,
                "Mean Abs Diff": np.nan
            })
            continue
            
        # 2. 데이터 병합 및 오차 계산
        df_orig = pd.read_csv(fp_orig, dtype={"oa_code": str})
        df_new = pd.read_csv(fp_new, dtype={"oa_code": str})
        
        comp = df_orig.merge(df_new, on="oa_code", suffixes=("_orig", "_new"))
        diff = (comp["accessibility_score_orig"] - comp["accessibility_score_new"]).abs()
        
        max_diff = diff.max()
        mean_diff = diff.mean()
        
        validation_records.append({
            "Scenario": tag,
            "Status": "완전 일치 (정상)" if max_diff < 1e-5 else "오차 발생 확인 필요",
            "Max Abs Diff": max_diff,
            "Mean Abs Diff": mean_diff
        })

# 3. 검증 결과 테이블 출력
df_val = pd.DataFrame(validation_records)
print("=" * 80)
print("             G2SFCA 원본 vs 리팩토링 코드 수치 검증 요약표")
print("=" * 80)
display(df_val)

             G2SFCA 원본 vs 리팩토링 코드 수치 검증 요약표


,Scenario,Status,Max Abs Diff,Mean Abs Diff
0,2021_week_오전_congested,완전 일치 (정상),0.0000,0.0000
1,2021_week_낮_normal,완전 일치 (정상),0.0000,0.0000
2,2021_weekend_오전_freeflow,완전 일치 (정상),0.0000,0.0000
3,2021_weekend_낮_normal,완전 일치 (정상),0.0000,0.0000
4,2021_week_심야_freeflow,완전 일치 (정상),0.0000,0.0000
5,2021_weekend_심야_freeflow,완전 일치 (정상),0.0000,0.0000
6,2022_week_오전_congested,완전 일치 (정상),0.0000,0.0000
7,2022_week_낮_normal,완전 일치 (정상),0.0000,0.0000
8,2022_weekend_오전_freeflow,완전 일치 (정상),0.0000,0.0000
9,2022_weekend_낮_normal,완전 일치 (정상),0.0000,0.0000
